# Lite-MDC — Reproduction Notebook (Keras / TensorFlow)

**Paper:** Advancing real-time plant disease detection: A lightweight deep learning approach and novel dataset for pigeon pea crop  
**Authors:** Sandesh Bhagat, Manesh Kokare, Vineet Haswani, Praful Hambarde, Trupti Taori, P.H. Ghante, D.K. Patil  
**Source:** Smart Agricultural Technology 7 (2024) 100408, Elsevier — DOI: 10.1016/j.atech.2024.100408

---

## Architecture: Lite-MDC (~2.2M params)
Lightweight CNN built on the **MDsConv** module (Multi-kernel Depthwise Separable Convolution):
`Channel Split -> [DWConv3x3 | DWConv5x5] -> Concat -> BN -> Conv1x1 (pointwise) -> BN -> LeakyReLU`.

Spatial downsampling is done via **stride 2 on the MDConv depthwise step** (blocks 1,3,6,9,12) — no MaxPooling.
The final layer (Conv_14) has **no LeakyReLU**. Classifier head = DwConv7x7 (global depthwise) -> PConv1x1 -> Conv1x1 -> GAP -> Softmax.

## Provenance of parameters
| Source | Parameters |
|---|---|
| Confirmed (paper) | Architecture (Table 5, 26 layers), MDsConv module (Algorithm 1, Fig 7/8), Adam, categorical cross-entropy, batch=16, input 224x224, ReduceLROnPlateau(factor=0.3, patience=5), stride-2 placement |
| User-confirmed | learning_rate=3e-4, epochs=100, LeakyReLU alpha=0.1, dropout schedule (0.1/0.2/0.3/0.4 in last 4 blocks), ImageNet mean/std normalization, dataset slug |

## Two scenarios (project rule)
- **Scenario 1 (Baseline):** train = Resize(224) + ImageNet normalize, **NO augmentation**.
- **Scenario 2 (Augmented):** train = Rotation + Zoom + HorizontalFlip + Brightness. Val/Test never augmented.


## 0. Setup

In [ ]:
!pip install -q openpyxl Pillow scikit-learn


In [ ]:

import os, random, json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, recall_score)
from sklearn.utils.class_weight import compute_class_weight

# ─── Reproducibility (project rule 7) ──────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─── Device detection ──────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow version:', tf.__version__)
print('GPUs available:', gpus)
if gpus:
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception as e:
            print('memory growth note:', e)

# ── Multi-GPU: MirroredStrategy (Rule 8) ──────────────────────────────────────
strategy = tf.distribute.MirroredStrategy()
n_gpus = strategy.num_replicas_in_sync
print(f"[Multi-GPU] MirroredStrategy: {n_gpus} GPU(s) available")


## 1. Configuration

In [ ]:
# ─── CONFIG ────────────────────────────────────────────────────────────────────
DATASET_PATH = '/kaggle/input/datasets/tminhhi/dataset-ot/dataset_smartcrop_plit_300'
WORK_DIR     = '/kaggle/working'

IMG_SIZE   = 224          # paper: 224x224x3
BATCH_SIZE = 16           # paper Mục 5.2
EPOCHS     = 100          # user-confirmed
LR         = 3e-4         # user-confirmed
LEAKY_ALPHA = 0.1         # user-confirmed

# ImageNet normalization (user-confirmed)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Augmentation params for Scenario 2 (paper Mục 5.2, ràng buộc 04 kỹ thuật)
AUG_ROTATION   = 30       # degrees
AUG_ZOOM       = 0.1      # giảm từ 0.2 → 0.1 (black border khi zoom-out quá lớn gây lỗi)
AUG_HFLIP      = True
AUG_BRIGHTNESS = 0.1      # giảm từ 0.4 → 0.1 (±10%, tránh clipping làm mất thông tin)

# Dropout schedule for last 4 blocks (user-confirmed)
#   after PConv_9 -> 0.1 | PConv_10 -> 0.2 | PConv_11 -> 0.3 | PConv_12 -> 0.4
DROPOUT_SCHEDULE = {9: 0.1, 10: 0.2, 11: 0.3, 12: 0.4}

print('Dataset path:', DATASET_PATH)
print('Path exists:', os.path.isdir(DATASET_PATH))
if os.path.isdir(DATASET_PATH):
    print('Contents:', os.listdir(DATASET_PATH))

## 2. Dataset Split
The user's dataset is **pre-split** into `train/ val/ test/`. We detect the layout and build
three `tf.data` pipelines. Paper split (60/20/20) is only used as a fallback if a single folder is found.


In [ ]:
# ─── Locate split folders ──────────────────────────────────────────────────────
_TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
_VAL_DIR   = os.path.join(DATASET_PATH, 'val')
_VALID_DIR = os.path.join(DATASET_PATH, 'valid')
_TEST_DIR  = os.path.join(DATASET_PATH, 'test')

has_test  = os.path.isdir(_TEST_DIR)
has_val   = os.path.isdir(_VAL_DIR)
has_valid = os.path.isdir(_VALID_DIR)

if has_val and has_test:
    TRAIN_DIR, VALID_EVAL_DIR, TEST_DIR = _TRAIN_DIR, _VAL_DIR, _TEST_DIR
    SPLIT_MODE = '3-folder (train/val/test)'
elif has_valid and has_test:
    TRAIN_DIR, VALID_EVAL_DIR, TEST_DIR = _TRAIN_DIR, _VALID_DIR, _TEST_DIR
    SPLIT_MODE = '3-folder (train/valid/test)'
else:
    raise RuntimeError('Expected pre-split train/val/test folders. Found: ' + str(os.listdir(DATASET_PATH)))

print('Split mode:', SPLIT_MODE)

# ─── class names auto-detect (project rule 12) ────────────────────────────────
class_names = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
NUM_CLASSES = len(class_names)
print('NUM_CLASSES (auto-detected):', NUM_CLASSES)
print('class_names:', class_names)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def _load_raw_ds(directory, shuffle):
    return keras.utils.image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='int',
        class_names=class_names,          # enforce consistent ordering across splits
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )

_mean_t = tf.constant(IMAGENET_MEAN.reshape(1,1,1,3))
_std_t  = tf.constant(IMAGENET_STD.reshape(1,1,1,3))

def normalize(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    x = (x - _mean_t) / _std_t
    return x, y

def make_baseline_pipeline(directory, shuffle):
    ds = _load_raw_ds(directory, shuffle)
    ds = ds.map(normalize, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

# ─── Scenario-2 augmentation layer (applied to TRAIN only) ─────────────────────
# fill_mode='nearest': lặp pixel biên thay vì zeros khi rotate/zoom-out
#   → tránh black borders trở thành giá trị âm lớn sau ImageNet normalize
# AUG_BRIGHTNESS=0.1: chỉ ±10% additive, tránh clipping mất thông tin ảnh
data_augment = keras.Sequential([
    layers.RandomRotation(AUG_ROTATION/360.0, fill_mode='nearest'),
    layers.RandomZoom(AUG_ZOOM, fill_mode='nearest'),
    layers.RandomFlip('horizontal'),
    layers.RandomBrightness(factor=AUG_BRIGHTNESS, value_range=(0.0,1.0)),
], name='augment')

def make_augmented_train_pipeline(directory):
    ds = _load_raw_ds(directory, shuffle=True)
    def aug_map(x, y):
        x = tf.cast(x, tf.float32) / 255.0
        x = data_augment(x, training=True)
        x = (x - _mean_t) / _std_t
        return x, y
    ds = ds.map(aug_map, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

# counts
def count_images(directory):
    total = 0
    per = {}
    for c in class_names:
        n = len([f for f in os.listdir(os.path.join(directory, c))
                 if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.gif','.tif','.tiff'))])
        per[c] = n
        total += n
    return total, per

n_train, train_per = count_images(TRAIN_DIR)
n_val,   _         = count_images(VALID_EVAL_DIR)
n_test,  _         = count_images(TEST_DIR)
print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Aug params: rotation=±{AUG_ROTATION}°, zoom=±{int(AUG_ZOOM*100)}%, brightness=±{int(AUG_BRIGHTNESS*100)}%, fill=nearest')

## 3. Dataset Distribution & Class Imbalance Check

In [ ]:
# ─── Class distribution bar chart + imbalance auto-detection (project rule 11) ──
counts = np.array([train_per[c] for c in class_names], dtype=np.float64)
max_c, min_c = counts.max(), counts.min()
imbalance_ratio = max_c / max(min_c, 1)
USE_CLASS_WEIGHT = imbalance_ratio >= 1.5

plt.figure(figsize=(10,5))
bars = plt.bar(class_names, counts, color='steelblue')
plt.title(f'Train class distribution (imbalance ratio = {imbalance_ratio:.2f})')
plt.ylabel('Number of images'); plt.xticks(rotation=30, ha='right')
for b, v in zip(bars, counts):
    plt.text(b.get_x()+b.get_width()/2, v, int(v), ha='center', va='bottom')
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR,'class_distribution.png'), dpi=300); plt.show()

print('Imbalance ratio = %.2f  ->  use class_weight: %s' % (imbalance_ratio, USE_CLASS_WEIGHT))

if USE_CLASS_WEIGHT:
    y_train_all = []
    for c_idx, c in enumerate(class_names):
        y_train_all += [c_idx]*int(train_per[c])
    y_train_all = np.array(y_train_all)
    cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train_all)
    CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
    print('class_weight:', CLASS_WEIGHT)
else:
    CLASS_WEIGHT = None
    print('Dataset balanced enough; no class weighting.')


## 4. Lite-MDC Model Builder
Implements Table 5 (26 layers). The channel sequence follows the **Output shape** column (authoritative per implement_plan): 64→64→64→128→128→256→256→256→256→512→512→512→512→512→512→512→512.

`build_mdsconv_block` = Channel Split → DW3x3 / DW5x5 (stride s) → Concat → BN → LeakyReLU → PointwiseConv(N) → BN → [LeakyReLU unless last] → optional Dropout.


In [ ]:
def leaky():
    return layers.LeakyReLU(alpha=LEAKY_ALPHA)

def mdsconv_block(x, out_channels, stride, name, dropout=0.0):
    """MDsConv: split -> [dw3x3 | dw5x5] -> concat -> BN -> LeakyReLU
                      -> pointwise(out_channels) -> BN -> LeakyReLU -> [Dropout]."""
    in_ch = x.shape[-1]
    half = in_ch // 2
    left  = layers.Lambda(lambda t: t[..., :half], name=name+'_split_l')(x)
    right = layers.Lambda(lambda t: t[..., half:], name=name+'_split_r')(x)
    left  = layers.DepthwiseConv2D(3, strides=stride, padding='same',
                                   use_bias=False, name=name+'_dw3')(left)
    right = layers.DepthwiseConv2D(5, strides=stride, padding='same',
                                   use_bias=False, name=name+'_dw5')(right)
    x = layers.Concatenate(name=name+'_concat')([left, right])
    x = layers.BatchNormalization(name=name+'_bn_dw')(x)
    x = leaky()(x)
    # pointwise
    x = layers.Conv2D(out_channels, 1, strides=1, padding='same',
                      use_bias=False, name=name+'_pw')(x)
    x = layers.BatchNormalization(name=name+'_bn_pw')(x)
    x = leaky()(x)
    if dropout and dropout > 0:
        x = layers.Dropout(dropout, name=name+'_drop')(x)
    return x

def build_lite_mdc(num_classes):
    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input')

    # Conv0: 3x3, 64, stride 1 -> BN -> LeakyReLU
    x = layers.Conv2D(64, 3, strides=1, padding='same', use_bias=False, name='conv0')(inp)
    x = layers.BatchNormalization(name='conv0_bn')(x)
    x = leaky()(x)

    # (out_channels, stride) per MDConv+PConv block, blocks 1..12 (Table 5, Output-shape column)
    # (PConv output channels, MDConv stride) — follows Table 5 'Output shape' column (authoritative)
    block_cfg = [
        (64,  2),   # blk1: MDConv_1 224->112, PConv_1 -> 64
        (128, 1),   # blk2: MDConv_2,          PConv_2 -> 128
        (128, 2),   # blk3: MDConv_3 112->56,  PConv_3 -> 128
        (256, 1),   # blk4: MDConv_4,          PConv_4 -> 256
        (256, 1),   # blk5: MDConv_5,          PConv_5 -> 256
        (256, 2),   # blk6: MDConv_6 56->28,   PConv_6 -> 256
        (512, 1),   # blk7: MDConv_7,          PConv_7 -> 512
        (512, 1),   # blk8: MDConv_8,          PConv_8 -> 512
        (512, 2),   # blk9: MDConv_9 28->14,   PConv_9 -> 512
        (512, 1),   # blk10: MDConv_10,        PConv_10 -> 512
        (512, 1),   # blk11: MDConv_11,        PConv_11 -> 512
        (512, 2),   # blk12: MDConv_12 14->7,  PConv_12 -> 512
    ]
    for i, (oc, st) in enumerate(block_cfg, start=1):
        dp = DROPOUT_SCHEDULE.get(i, 0.0)
        x = mdsconv_block(x, out_channels=oc, stride=st, name=f'block{i}', dropout=dp)

    # DwConv_13: depthwise 7x7 on 7x7 feature map (valid) -> 1x1x512
    x = layers.DepthwiseConv2D(7, strides=1, padding='valid', use_bias=False, name='dwconv13')(x)
    x = layers.BatchNormalization(name='dwconv13_bn')(x)
    x = leaky()(x)
    # PConv_13: pointwise 1x1 512
    x = layers.Conv2D(512, 1, padding='same', use_bias=False, name='pconv13')(x)
    x = layers.BatchNormalization(name='pconv13_bn')(x)
    x = leaky()(x)
    # Conv_14: pointwise 1x1 512 (replaces FC) -> BN -> NO activation (last layer)
    x = layers.Conv2D(512, 1, padding='same', use_bias=False, name='conv14')(x)
    x = layers.BatchNormalization(name='conv14_bn')(x)
    # (no LeakyReLU here, per paper)

    x = layers.GlobalAveragePooling2D(name='gap')(x)
    out = layers.Dense(num_classes, activation='softmax', name='softmax')(x)
    return keras.Model(inp, out, name='Lite_MDC')

MODEL_NAME = 'Lite-MDC'


## 5. Model Summary (param count target ~2.2M)

In [ ]:
_probe = build_lite_mdc(NUM_CLASSES)
_probe.summary()
_total = _probe.count_params()
print(f'\nTotal params: {_total:,}  (paper target ~2.2M)')
del _probe


## 6. Evaluation Helpers

In [ ]:
def get_true_pred(model, ds):
    y_true, y_pred = [], []
    for xb, yb in ds:
        p = model.predict(xb, verbose=0)
        y_pred.extend(np.argmax(p, axis=1).tolist())
        y_true.extend(yb.numpy().tolist())
    return np.array(y_true), np.array(y_pred)

def compute_metrics(y_true, y_pred, total_params):
    return {
        'model_name'  : MODEL_NAME,
        'accuracy'    : accuracy_score(y_true, y_pred),
        'f1_macro'    : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'total_params': int(total_params),
    }

def plot_learning_curve(history, title, save_path):
    h = history.history
    fig, ax = plt.subplots(1, 2, figsize=(14,5))
    ax[0].plot(h['loss'], label='train'); ax[0].plot(h.get('val_loss',[]), label='val')
    ax[0].set_title(title+' — Loss'); ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].legend()
    acc_k = 'accuracy' if 'accuracy' in h else 'acc'
    vacc_k = 'val_accuracy' if 'val_accuracy' in h else 'val_acc'
    ax[1].plot(h.get(acc_k,[]), label='train'); ax[1].plot(h.get(vacc_k,[]), label='val')
    ax[1].set_title(title+' — Accuracy'); ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy'); ax[1].legend()
    plt.tight_layout(); plt.savefig(save_path, dpi=300); plt.show()

def plot_confusion(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(8,7))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(class_names, rotation=45, ha='right'); ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    thresh = cm.max()/2 if cm.max() else 0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, int(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > thresh else 'black')
    fig.colorbar(im); plt.tight_layout(); plt.savefig(save_path, dpi=300); plt.show()

def make_callbacks(ckpt_path):
    return [
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor='val_accuracy',
                                        save_best_only=True, mode='max', verbose=1,
                                        save_weights_only=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                                          patience=5, verbose=1),
        # Dừng sớm nếu val_accuracy không cải thiện sau 15 epoch liên tiếp.
        # patience=15 > ReduceLROnPlateau patience=5: cho model đủ thời gian
        # adapt với LR mới trước khi kết luận đã converge.
        # restore_best_weights=False vì ModelCheckpoint đã lưu best checkpoint.
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15,
                                      mode='max', restore_best_weights=False,
                                      verbose=1),
    ]

---
# Scenario 1 — Baseline (No Augmentation)
Train transform = Resize(224) + ImageNet normalize only. No augmentation. Val/Test identical.


In [ ]:

tf.keras.backend.clear_session()
tf.random.set_seed(SEED); np.random.seed(SEED)

train_ds_s1 = make_baseline_pipeline(TRAIN_DIR, shuffle=True)
val_ds_s1   = make_baseline_pipeline(VALID_EVAL_DIR, shuffle=False)
test_ds_s1  = make_baseline_pipeline(TEST_DIR, shuffle=False)

# Wrap model creation in strategy.scope() for multi-GPU (Rule 8)
with strategy.scope():
    model_s1 = build_lite_mdc(NUM_CLASSES)
    model_s1.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

# .weights.h5 = weights-only format (avoids Lambda serialization error in Keras 3)
CKPT_S1 = os.path.join(WORK_DIR, 'best_model_s1.weights.h5')
print('Scenario 1 model ready. Total params:', f'{model_s1.count_params():,}')


In [ ]:
history_s1 = model_s1.fit(
    train_ds_s1,
    validation_data=val_ds_s1,
    epochs=EPOCHS,
    class_weight=CLASS_WEIGHT,
    callbacks=make_callbacks(CKPT_S1),
    verbose=1,
)


In [ ]:
# Reload BEST checkpoint before test eval (project rule 10)
# load_weights() instead of load_model() — Lambda layers can't be deserialized in Keras 3
model_s1.load_weights(CKPT_S1)
y_true_s1, y_pred_s1 = get_true_pred(model_s1, test_ds_s1)
baseline_metrics = compute_metrics(y_true_s1, y_pred_s1, model_s1.count_params())
print('Baseline test metrics:', baseline_metrics)

### S1 — Classification Report

In [ ]:
print('Scenario 1 (Baseline) — Classification Report\n')
print(classification_report(y_true_s1, y_pred_s1, target_names=class_names, digits=4, zero_division=0))


### S1 — Confusion Matrix

In [ ]:
BASELINE_CONFUSION_MATRIX_PATH = os.path.join(WORK_DIR, 'cm_baseline.png')
plot_confusion(y_true_s1, y_pred_s1, 'Lite-MDC Baseline — Confusion Matrix', BASELINE_CONFUSION_MATRIX_PATH)


### S1 — Learning Curve

In [ ]:
BASELINE_LEARNING_CURVE_PATH = os.path.join(WORK_DIR, 'lc_baseline.png')
plot_learning_curve(history_s1, 'Lite-MDC Baseline', BASELINE_LEARNING_CURVE_PATH)


In [ ]:
# Free GPU memory before Scenario 2 (project rule 13)
import gc
del model_s1, train_ds_s1, val_ds_s1, test_ds_s1
gc.collect()
tf.keras.backend.clear_session()


---
# Scenario 2 — Augmented
Train transform adds 4 augmentations: **Rotation (±30°, fill=nearest), Zoom (±10%, fill=nearest), Horizontal Flip, Brightness (±10%)**.
Validation/Test use baseline transform only (no augmentation).

In [ ]:

tf.random.set_seed(SEED); np.random.seed(SEED)

train_ds_s2 = make_augmented_train_pipeline(TRAIN_DIR)
val_ds_s2   = make_baseline_pipeline(VALID_EVAL_DIR, shuffle=False)
test_ds_s2  = make_baseline_pipeline(TEST_DIR, shuffle=False)

# Wrap model creation in strategy.scope() for multi-GPU (Rule 8)
with strategy.scope():
    model_s2 = build_lite_mdc(NUM_CLASSES)
    model_s2.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
# .weights.h5 = weights-only format (avoids Lambda serialization error in Keras 3)
CKPT_S2 = os.path.join(WORK_DIR, 'best_model_s2.weights.h5')
print('Scenario 2 model ready. Total params:', f'{model_s2.count_params():,}')


In [ ]:
history_s2 = model_s2.fit(
    train_ds_s2,
    validation_data=val_ds_s2,
    epochs=EPOCHS,
    class_weight=CLASS_WEIGHT,
    callbacks=make_callbacks(CKPT_S2),
    verbose=1,
)


In [ ]:
# Reload BEST checkpoint before test eval (project rule 10)
# load_weights() instead of load_model() — Lambda layers can't be deserialized in Keras 3
model_s2.load_weights(CKPT_S2)
y_true_s2, y_pred_s2 = get_true_pred(model_s2, test_ds_s2)
aug_metrics = compute_metrics(y_true_s2, y_pred_s2, model_s2.count_params())
print('Augmented test metrics:', aug_metrics)

### S2 — Classification Report

In [ ]:
print('Scenario 2 (Augmented) — Classification Report\n')
print(classification_report(y_true_s2, y_pred_s2, target_names=class_names, digits=4, zero_division=0))


### S2 — Confusion Matrix

In [ ]:
AUG_CONFUSION_MATRIX_PATH = os.path.join(WORK_DIR, 'cm_augmented.png')
plot_confusion(y_true_s2, y_pred_s2, 'Lite-MDC Augmented — Confusion Matrix', AUG_CONFUSION_MATRIX_PATH)


### S2 — Learning Curve

In [ ]:
AUG_LEARNING_CURVE_PATH = os.path.join(WORK_DIR, 'lc_augmented.png')
plot_learning_curve(history_s2, 'Lite-MDC Augmented', AUG_LEARNING_CURVE_PATH)


---
# Results Export — results_summary.xlsx
Two sheets (**Baseline**, **Augmented**) with embedded Learning Curve + Confusion Matrix images.


In [ ]:
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

EXCEL_OUTPUT_PATH = os.path.join(WORK_DIR, 'results_summary.xlsx')

HEADERS = ['STT','Tên Model','Accuracy','F1-score','Recall',
           'Learning Curve','Confusion Matrix','Parameters']
COL_WIDTHS = [6,35,12,12,12,38,38,18]
IMG_ROW_HEIGHT = 200

def style_header_row(ws):
    fill = PatternFill('solid', fgColor='1F4E79')
    font = Font(bold=True, color='FFFFFF', size=11)
    border = Border(bottom=Side(style='medium', color='FFFFFF'), right=Side(style='thin', color='FFFFFF'))
    for col, header in enumerate(HEADERS, start=1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.fill, cell.font, cell.border = fill, font, border
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def write_result_row(ws, row_num, stt, metrics, lc_path, cm_path):
    data = [stt, metrics['model_name'], round(metrics['accuracy'],4),
            round(metrics['f1_macro'],4), round(metrics['recall_macro'],4),
            '', '', f"{metrics['total_params']:,}"]
    for col, val in enumerate(data, start=1):
        c = ws.cell(row=row_num, column=col, value=val)
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    ws.row_dimensions[row_num].height = IMG_ROW_HEIGHT
    if os.path.exists(lc_path):
        img = XLImage(lc_path); img.width, img.height = 280,190; ws.add_image(img, f'F{row_num}')
    if os.path.exists(cm_path):
        img = XLImage(cm_path); img.width, img.height = 280,190; ws.add_image(img, f'G{row_num}')

def build_sheet(wb, name, metrics, lc_path, cm_path):
    ws = wb.create_sheet(title=name)
    for col, w in enumerate(COL_WIDTHS, start=1):
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[1].height = 30
    style_header_row(ws)
    write_result_row(ws, 2, 1, metrics, lc_path, cm_path)
    return ws

wb = Workbook(); wb.remove(wb.active)
build_sheet(wb, 'Baseline', baseline_metrics, BASELINE_LEARNING_CURVE_PATH, BASELINE_CONFUSION_MATRIX_PATH)
build_sheet(wb, 'Augmented', aug_metrics, AUG_LEARNING_CURVE_PATH, AUG_CONFUSION_MATRIX_PATH)
wb.save(EXCEL_OUTPUT_PATH)
print('[Excel] saved:', EXCEL_OUTPUT_PATH)
print(f"  Baseline  -> Acc={baseline_metrics['accuracy']:.4f} | F1={baseline_metrics['f1_macro']:.4f} | Recall={baseline_metrics['recall_macro']:.4f} | Params={baseline_metrics['total_params']:,}")
print(f"  Augmented -> Acc={aug_metrics['accuracy']:.4f} | F1={aug_metrics['f1_macro']:.4f} | Recall={aug_metrics['recall_macro']:.4f} | Params={aug_metrics['total_params']:,}")
